# P1 - Classificacio Baseline

Objectiu: construir una baseline simple i interpretable per veure si el dataset ja permet una classificació mínimament defensable.

## Ruta de treball

1. Triar una estratègia simple.
2. Aplicar-la a tots els tickets.
3. Comparar predicció i etiqueta esperada.
4. Escriure on falla i què exigiria una aproximació millor.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'dades' / 'tickets_helpdesk.csv'
df = pd.read_csv(DATA)
df[['ticket_id', 'text_ticket', 'departament_sospitat']].head()

## 1. Baseline per paraules clau

No és un model de producció, però és una forma útil de fer visible la lògica inicial.

In [ ]:
rules = {
    'identitat': ['contrasenya', 'login', 'blocked', 'mfa', 'codi'],
    'xarxes': ['vpn', 'certificat', '403', 'oficina'],
    'correu': ['correu', 'quota', 'imap', 'adjunts'],
    'aplicacions': ['aplicacio', 'modul', 'panell', 'informes'],
}

def predict_department(text):
    text = text.lower()
    scores = {label: sum(word in text for word in words) for label, words in rules.items()}
    best_label = max(scores, key=scores.get)
    if scores[best_label] == 0:
        return 'sense_regla'
    return best_label

df['baseline_pred'] = df['text_ticket'].apply(predict_department)
df[['ticket_id', 'departament_sospitat', 'baseline_pred', 'text_ticket']]

## 2. Comparació bàsica

Açò no substituïx una mètrica formal, però ja ajuda a veure si la baseline està alineada amb les etiquetes.

In [ ]:
df['correct'] = df['baseline_pred'] == df['departament_sospitat']
df['correct'].value_counts().rename_axis('correcte').to_frame('tickets')

In [ ]:
errors = df[~df['correct']][['ticket_id', 'departament_sospitat', 'baseline_pred', 'text_ticket']]
errors if not errors.empty else 'La baseline ha encertat tots els casos, cosa poc probable en un entorn real.'

## 3. Llegir els errors

Els errors interessen més que els encerts, perquè indiquen on una baseline per paraules clau es queda curta.

In [ ]:
df.groupby(['departament_sospitat', 'baseline_pred']).size().rename('count').reset_index().sort_values('count', ascending=False)

## 4. Conclusió docent

Tanca amb tres idees:

- quins departaments es detecten millor amb regles simples
- quins casos exigirien text vectoritzat o un model supervisat
- per què esta baseline encara no és suficient per a una API fiable